In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from pathlib import Path
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
if iskaggle:
    path = Path('../input/nlp-getting-started')

In [ ]:
pip show datasets

In [ ]:
!ls {path}

In [ ]:
import pandas as pd
train_df = pd.read_csv(path/'train.csv')
test_df = pd.read_csv(path/'test.csv')
train_df

In [ ]:
train_df.describe(include='object')

In [ ]:
train = train_df.drop(['id', 'target'], axis=1)
test = test_df.drop(['id'], axis=1)

In [ ]:
train

In [ ]:
test

In [ ]:
train_list = train.to_dict(orient='records')
test_list = test.to_dict(orient='records')
train_list[:3]

In [ ]:
test_list[:2]

In [ ]:
train_text = [str(row) for row in train_list]
test_text = [str(row) for row in test_list]
train_text[:3]

In [ ]:
 test_text[:3]

In [ ]:
train_df['input'] = train_text
test_df['input'] = test_text
train_df.input.head()

In [ ]:
 test_df.input.head()

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
from datasets import Dataset,DatasetDict

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
train_dataset

In [ ]:
test_dataset = Dataset.from_pandas(test_df)
test_dataset

In [ ]:
model_nm = 'microsoft/deberta-v3-small'

In [ ]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)

For the first I used a random sentence generator and for the second I used a list of dinosaur names.

In [ ]:
# Prac TODO: try different text!!!!! -> Done
tokz.tokenize("Kevin embraced his ability to be at the wrong place at the wrong time.")

In [ ]:
# Prac TODO: try different text!!!!! -> Done
tokz.tokenize("Tyrannosaurus Rex, Triceratops, Stegosaurus, Brachiosaurus, Velociraptor, and Ankylosaurus")

In [ ]:
def tok_func(x): return tokz(x["input"])

In [ ]:
train_tok_dataset = train_dataset.map(tok_func, batched=True)

In [ ]:
test_tok_dataset = test_dataset.map(tok_func, batched=True)

In [ ]:
row = train_tok_dataset[0]
row['input'], row['input_ids']

In [ ]:
row = test_tok_dataset[0]
row['input'], row['input_ids']

In [ ]:
train_tok_dataset = train_tok_dataset.rename_columns({'target':'labels'})

Example code uses 0.25 split, that leaves it as 5709 - 1904.
Seems to close for the number of entries so I'm lowering it to a 0.05 split.

In [ ]:
dds = train_tok_dataset.train_test_split(0.05, seed=42)
dds

fastai-lesson4-Getting-started-NLP-v250619 says that I also need to make a 'test set' using the following commented out code. However, after checking the solution it appears that it isn't nessesary for this task?

In [ ]:
# eval_df['input'] = 'TEXT1: ' + eval_df.context + '; TEXT2: ' + eval_df.target + '; ANC1: ' + eval_df.anchor
# eval_ds = Dataset.from_pandas(eval_df).map(tok_func, batched=True)

In [ ]:
from transformers import TrainingArguments,Trainer
bs = 32
epochs = 1
lr = 8e-5

In [ ]:
args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, 
                         lr_scheduler_type='cosine', 
                         fp16=True,
    eval_strategy="epoch", 
                         per_device_train_batch_size=bs, 
                         per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, 
                         weight_decay=0.01, 
                         report_to='none')

In [ ]:
from sklearn.metrics import f1_score

# def my_metric(eval_pred): 
#     return {'F1': f1_score(*eval_pred, zero_division=0)}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)

trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'], tokenizer=tokz)

In [ ]:
trainer.train();

In [ ]:
preds = trainer.predict(test_tok_dataset)
preds

In [ ]:
preds.predictions

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)
class_preds

In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub

In [ ]:
sub['target'] = class_preds
sub

In [ ]:
# 1 epoch = LB 0.82592 'microsoft/deberta-v3-small' 1 epochs
name = os.path.basename(model_nm)
sub.to_csv(f'submission_{name}_ep{epochs}.csv', index=False)  
sub.to_csv(f'submission.csv', index=False)  